In [ ]:
!pip install -q transformers peft trl datasets accelerate bitsandbytes huggingface_hub pillow

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from huggingface_hub import login
login()

In [ ]:
import pandas as pd
import os

DATA_DIR = "/content/drive/MyDrive/synthetic_cards_data"
CSV_PATH = "/content/drive/MyDrive/synthetic_cards_progress.csv"

df = pd.read_csv(CSV_PATH)

print("Total Records:", len(df))
display(df.head())

In [ ]:
from datasets import Dataset, Image

df = df[
    [
        "image_path",
        "doc_type",
        "side",
        "state",
    ]
].copy()

missing = df["image_path"].apply(lambda x: not os.path.exists(x)).sum()

print(f"Missing Images : {missing}")
print(f"Available Images : {len(df) - missing}")

assert missing == 0, "Some image paths are invalid!"

hf_dataset = Dataset.from_pandas(
    df,
    preserve_index=False,
)

hf_dataset = hf_dataset.cast_column(
    "image_path",
    Image()
)

print(hf_dataset)

In [ ]:
from datasets import DatasetDict

dataset = hf_dataset.train_test_split(
    test_size=0.20,
    seed=42,
    shuffle=True,
)

train_dataset = dataset["train"]
test_dataset = dataset["test"]

print(f"Training Images : {len(train_dataset)}")
print(f"Testing Images  : {len(test_dataset)}")

In [ ]:
doc_type_to_id = {
    label: idx
    for idx, label in enumerate(
        sorted(train_dataset.unique("doc_type"))
    )
}

side_to_id = {
    label: idx
    for idx, label in enumerate(
        sorted(train_dataset.unique("side"))
    )
}

state_to_id = {
    label: idx
    for idx, label in enumerate(
        sorted(train_dataset.unique("state"))
    )
}


id_to_doc_type = {
    idx: label
    for label, idx in doc_type_to_id.items()
}

id_to_side = {
    idx: label
    for label, idx in side_to_id.items()
}

id_to_state = {
    idx: label
    for label, idx in state_to_id.items()
}


NUM_DOC_TYPES = len(doc_type_to_id)
NUM_SIDES = len(side_to_id)
NUM_STATES = len(state_to_id)


print("Document Types :", NUM_DOC_TYPES)
print("Sides          :", NUM_SIDES)
print("States         :", NUM_STATES)

In [ ]:
from transformers import AutoImageProcessor

MODEL_NAME = "google/vit-base-patch16-224"

processor = AutoImageProcessor.from_pretrained(MODEL_NAME)

print("Processor Loaded Successfully")

In [ ]:
import torch

def preprocess_batch(batch):
    images = batch["image_path"]

    processed = processor(
        images=images,
        return_tensors="pt"
    )

    return {
        "pixel_values": processed["pixel_values"],
        "doc_type_labels": [
            doc_type_to_id[x] for x in batch["doc_type"]
        ],
        "side_labels": [
            side_to_id[x] for x in batch["side"]
        ],
        "state_labels": [
            state_to_id[x] for x in batch["state"]
        ],
    }

In [ ]:
from datasets import load_from_disk
import os

CACHE_PATH = "/content/drive/MyDrive/vit_processed_cards"

if os.path.exists(CACHE_PATH):
    print("Loading cached dataset...")
    dataset = load_from_disk(CACHE_PATH)

else:
    print("Preprocessing images...")

    dataset = DatasetDict({
        "train": train_dataset,
        "test": test_dataset,
    })

    dataset = dataset.map(
        preprocess_batch,
        batched=True,
        batch_size=32,
        remove_columns=dataset["train"].column_names,
        desc="Preprocessing Images",
    )

    dataset.save_to_disk(CACHE_PATH)
    print("Dataset cached successfully!")

In [ ]:
dataset.set_format(
    type="torch",
    columns=[
        "pixel_values",
        "doc_type_labels",
        "side_labels",
        "state_labels",
    ]
)

train_dataset = dataset["train"]
test_dataset = dataset["test"]

print(train_dataset[0].keys())
print(train_dataset[0]["pixel_values"].shape)

In [ ]:
from torch.utils.data import DataLoader

BATCH_SIZE = 32

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True,
    persistent_workers=True,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True,
    persistent_workers=True,
)

print(len(train_loader), len(test_loader))

In [ ]:
import torch
import torch.nn as nn

from transformers import ViTModel


class MultiHeadViT(nn.Module):

    def __init__(self):

        super().__init__()

        self.backbone = ViTModel.from_pretrained(
            MODEL_NAME
        )

        hidden = self.backbone.config.hidden_size

        self.dropout = nn.Dropout(0.1)

        self.doc_classifier = nn.Linear(
            hidden,
            NUM_DOC_TYPES,
        )

        self.side_classifier = nn.Linear(
            hidden,
            NUM_SIDES,
        )

        self.state_classifier = nn.Linear(
            hidden,
            NUM_STATES,
        )

    def forward(self, pixel_values):

        outputs = self.backbone(
            pixel_values=pixel_values
        )

        cls = outputs.last_hidden_state[:, 0]

        cls = self.dropout(cls)

        return {
            "doc": self.doc_classifier(cls),
            "side": self.side_classifier(cls),
            "state": self.state_classifier(cls),
        }

In [ ]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

model = MultiHeadViT().to(device)

print(device)

In [ ]:
from transformers import get_cosine_schedule_with_warmup

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=2e-5,
    weight_decay=0.01,
)

EPOCHS = 10

total_steps = len(train_loader) * EPOCHS

scheduler = get_cosine_schedule_with_warmup(
    optimizer,
    num_warmup_steps=int(0.1 * total_steps),
    num_training_steps=total_steps,
)

In [ ]:
from torch.amp import GradScaler, autocast

scaler = GradScaler("cuda")

In [ ]:
import torch

def evaluate(model, dataloader, criterion):

    model.eval()

    total_loss = 0

    doc_correct = 0
    side_correct = 0
    state_correct = 0
    combined_correct = 0

    total = 0

    with torch.no_grad():

        for batch in dataloader:

            pixel_values = batch["pixel_values"].to(device)

            doc_labels = batch["doc_type_labels"].to(device)
            side_labels = batch["side_labels"].to(device)
            state_labels = batch["state_labels"].to(device)

            outputs = model(pixel_values)

            doc_loss = criterion(outputs["doc"], doc_labels)
            side_loss = criterion(outputs["side"], side_labels)
            state_loss = criterion(outputs["state"], state_labels)

            loss = doc_loss + side_loss + state_loss

            total_loss += loss.item()

            doc_pred = outputs["doc"].argmax(1)
            side_pred = outputs["side"].argmax(1)
            state_pred = outputs["state"].argmax(1)

            doc_correct += (doc_pred == doc_labels).sum().item()
            side_correct += (side_pred == side_labels).sum().item()
            state_correct += (state_pred == state_labels).sum().item()

            combined_correct += (
                (doc_pred == doc_labels) &
                (side_pred == side_labels) &
                (state_pred == state_labels)
            ).sum().item()

            total += len(doc_labels)

    return {
        "loss": total_loss / len(dataloader),
        "doc_acc": 100 * doc_correct / total,
        "side_acc": 100 * side_correct / total,
        "state_acc": 100 * state_correct / total,
        "combined_acc": 100 * combined_correct / total,
    }

In [ ]:
from tqdm.auto import tqdm

def train_one_epoch(model, dataloader):

    model.train()

    running_loss = 0

    progress = tqdm(
        dataloader,
        leave=False,
        desc="Training"
    )

    for batch in progress:

        pixel_values = batch["pixel_values"].to(device)

        doc_labels = batch["doc_type_labels"].to(device)
        side_labels = batch["side_labels"].to(device)
        state_labels = batch["state_labels"].to(device)

        optimizer.zero_grad(set_to_none=True)

        with autocast("cuda"):

            outputs = model(pixel_values)

            doc_loss = criterion(outputs["doc"], doc_labels)
            side_loss = criterion(outputs["side"], side_labels)
            state_loss = criterion(outputs["state"], state_labels)

            loss = doc_loss + side_loss + state_loss

        scaler.scale(loss).backward()

        scaler.unscale_(optimizer)

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            1.0
        )

        scaler.step(optimizer)
        scaler.update()

        scheduler.step()

        running_loss += loss.item()

        progress.set_postfix(
            loss=f"{loss.item():.4f}"
        )

    return running_loss / len(dataloader)

In [ ]:
import os

CHECKPOINT = "/content/drive/MyDrive/multihead_vit_checkpoint.pth"
BEST_MODEL = "/content/drive/MyDrive/multihead_vit_best.pth"

start_epoch = 0
best_accuracy = 0

if os.path.exists(CHECKPOINT):

    checkpoint = torch.load(
        CHECKPOINT,
        map_location=device
    )

    model.load_state_dict(
        checkpoint["model"]
    )

    optimizer.load_state_dict(
        checkpoint["optimizer"]
    )

    scheduler.load_state_dict(
        checkpoint["scheduler"]
    )

    scaler.load_state_dict(
        checkpoint["scaler"]
    )

    start_epoch = checkpoint["epoch"] + 1
    best_accuracy = checkpoint["best_accuracy"]

    print(f"Resuming from epoch {start_epoch}")

In [ ]:
EARLY_STOPPING = 5

patience = 0

for epoch in range(start_epoch, EPOCHS):

    print("=" * 60)
    print(f"Epoch {epoch+1}/{EPOCHS}")

    train_loss = train_one_epoch(
        model,
        train_loader,
    )

    metrics = evaluate(
        model,
        test_loader,
        criterion,
    )

    print(f"Train Loss      : {train_loss:.4f}")
    print(f"Validation Loss : {metrics['loss']:.4f}")

    print(f"Doc Accuracy    : {metrics['doc_acc']:.2f}%")
    print(f"Side Accuracy   : {metrics['side_acc']:.2f}%")
    print(f"State Accuracy  : {metrics['state_acc']:.2f}%")
    print(f"Combined Acc    : {metrics['combined_acc']:.2f}%")

    torch.save(
        {
            "epoch": epoch,
            "model": model.state_dict(),
            "optimizer": optimizer.state_dict(),
            "scheduler": scheduler.state_dict(),
            "scaler": scaler.state_dict(),
            "best_accuracy": best_accuracy,
        },
        CHECKPOINT,
    )

    if metrics["combined_acc"] > best_accuracy:

        best_accuracy = metrics["combined_acc"]

        torch.save(
            model.state_dict(),
            BEST_MODEL,
        )

        print("✅ Best model updated.")

        patience = 0

    else:

        patience += 1

        if patience >= EARLY_STOPPING:

            print("Early stopping.")

            break

In [ ]:
model.load_state_dict(
    torch.load(BEST_MODEL, map_location=device)
)

model.eval()

print("Best model loaded.")

In [ ]:
import json

label_mappings = {
    "doc_type_to_id": doc_type_to_id,
    "side_to_id": side_to_id,
    "state_to_id": state_to_id,
}

with open("/content/label_mappings.json", "w") as f:
    json.dump(label_mappings, f, indent=4)

print("Label mappings saved.")

In [ ]:
from PIL import Image
import torch

# Reverse mappings
id_to_doc_type = {v: k for k, v in doc_type_to_id.items()}
id_to_side = {v: k for k, v in side_to_id.items()}
id_to_state = {v: k for k, v in state_to_id.items()}


def predict(image_path):

    image = Image.open(image_path).convert("RGB")

    inputs = processor(
        images=image,
        return_tensors="pt"
    )

    pixel_values = inputs["pixel_values"].to(device)

    with torch.no_grad():
        outputs = model(pixel_values)

    doc = outputs["doc"].argmax(1).item()
    side = outputs["side"].argmax(1).item()
    state = outputs["state"].argmax(1).item()

    print("=" * 40)
    print("Prediction")
    print("=" * 40)
    print("Document Type :", id_to_doc_type[doc])
    print("Side          :", id_to_side[side])
    print("State         :", id_to_state[state])

    return {
        "doc_type": id_to_doc_type[doc],
        "side": id_to_side[side],
        "state": id_to_state[state]
    }

In [ ]:
import matplotlib.pyplot as plt

def predict_and_show(image_path):

    image = Image.open(image_path).convert("RGB")

    plt.figure(figsize=(6,6))
    plt.imshow(image)
    plt.axis("off")

    prediction = predict(image_path)

    plt.title(
        f"{prediction['doc_type']} | "
        f"{prediction['side']} | "
        f"{prediction['state']}"
    )

    plt.show()

In [ ]:
predict_and_show(df.iloc[5]["image_path"])